# Problem 2: Transmembrane Protein Modeling

## Problem Summary
Model a transmembrane protein structure, validate it, and determine if it functions as monomer, dimer, or tetramer.

## Biological Context

**What are transmembrane proteins?**
- Proteins that span the lipid bilayer membrane
- Have hydrophobic regions (~20 aa) that interact with lipid tails
- Two main structures:
  - **Alpha-helical bundles**: Most common, e.g., GPCRs, ion channels, aquaporins
  - **Beta-barrels**: Found in outer membranes of bacteria, mitochondria, chloroplasts

**Why TM proteins are special:**
- Different energy landscape than soluble proteins
- Hydrophobic residues are FAVORED in TM regions (opposite to soluble proteins)
- Often form oligomers within the membrane
- Mediate signal transduction, transport, adhesion

**Special considerations for modeling:**
- Need TM-specific validation (ProSA is NOT appropriate)
- Must consider membrane environment
- Oligomeric state often essential for function

## Points Distribution
- a) Fold identification: 0.5 pts
- b) Function and PFAM family: 0.5 pts
- c) Structure modeling: 0.5 pts
- d) ProSA validation discussion: 0.5 pts
- e) Oligomeric state modeling: 1.0 pts
- f) Fix structural problems: 1.0 pts

**Total: 4 points**

---
## Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Modify these paths as needed
# ============================================================

# Input: The given sequence from problem_2.txt
# Read the sequence from the problem file
PROBLEM_FILE = "Exam/problem_2.txt"

# Output directory and file naming prefix
OUTPUT_DIR = "Problem_2_outputs"
OUTPUT_PREFIX = "p42"  # Files will be named p42c.pdb, p42d.png, etc.

# Database paths (local databases)
SWISSPROT_DB = "databases/swissprot/swissprot"
PDBAA_DB = "databases/pdb_seq/pdbaa"
PDBAA_FASTA = "databases/pdb_seq/pdbaa.fasta"
PFAM_DB = "databases/hmm/Pfam/Pfam-A.hmm"

# Working directories
TEMP_DIR = "temp"
TEMPLATES_DIR = "Templates"
ALIGNMENTS_DIR = "Alignments"
MODELLER_DIR = "Modeller_Templates"

In [ ]:
import os
import sys
import re
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd()))

# Import project modules
from src.pipeline import HomologyPipeline, run_pipeline
from src.Homology.retrieve import TemplateRetriever
from src.Homology.domains import TemplateProcessor
from src.Analysis.assessment import (
    identify_protein_family, 
    create_hmm_profile, 
    run_dssp_analysis,
    validate_model_regions,
    fix_model_problems,
    ModelAssessor
)
from src.modeller.scripts import (
    generate_single_template_script,
    generate_multi_template_script,
    generate_loop_refinement_script,
    create_modeller_scripts_for_pipeline,
    ModellerRunner
)
from src.UI.app import CoverageVisualizer, DomainVisualizer

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.PDB import PDBParser, PDBIO, PPBuilder

# Create output directories
for d in [OUTPUT_DIR, TEMP_DIR, TEMPLATES_DIR, ALIGNMENTS_DIR, MODELLER_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Output prefix: {OUTPUT_PREFIX}")

In [ ]:
# Read sequence from problem file
def read_sequence_from_file(filepath):
    """Extract sequence from problem file (may be FASTA or plain text)."""
    with open(filepath) as f:
        content = f.read()
    
    # Try FASTA format first
    if content.startswith('>'):
        lines = content.strip().split('\n')
        seq = ''.join(line.strip() for line in lines[1:] if not line.startswith('>'))
    else:
        # Plain sequence
        seq = ''.join(content.split())
    
    # Clean sequence
    seq = ''.join(c for c in seq.upper() if c.isalpha())
    return seq

TARGET_SEQUENCE = read_sequence_from_file(PROBLEM_FILE)
TARGET_NAME = "TM_Protein"

print(f"Sequence length: {len(TARGET_SEQUENCE)} residues")
print(f"\nSequence:")
print(TARGET_SEQUENCE)

In [ ]:
# Save target sequence as FASTA
TARGET_FASTA = Path(OUTPUT_DIR) / "target.fa"

record = SeqRecord(Seq(TARGET_SEQUENCE), id=TARGET_NAME, description="Transmembrane protein")
SeqIO.write([record], TARGET_FASTA, "fasta")

print(f"Saved target sequence to: {TARGET_FASTA}")

---
## a) Fold Identification (0.5 pts)

### Biological Background: TM Protein Folds

**Common TM protein folds:**
1. **7-TM bundle (GPCRs)**: G protein-coupled receptors, ~800 in human genome
2. **4-helix bundle**: Simple channels, some cytokine receptors
3. **Aquaporin fold (6+2 TM)**: Water channels, form tetramers
4. **Ion channel folds**: Variable, often 6 TM per subunit
5. **ABC transporter**: 12 TM helices typically
6. **Beta-barrel (8-22 strands)**: Porins, outer membrane proteins

**How to identify TM fold:**
1. Count number of TM helices (hydrophobicity plot)
2. BLAST against PDB - what structures match?
3. Domain analysis - Pfam will identify TM families

**Key observation for aquaporins:**
- MIP (Major Intrinsic Protein) family
- 6 TM helices + 2 half-helices (re-entrant loops)
- Form functional tetramers

In [ ]:
# Method 1: Hydrophobicity analysis to predict TM helices
def predict_tm_regions(sequence, window_size=19, hydrophobicity_threshold=0.4):
    """
    Predict TM regions using Kyte-Doolittle hydrophobicity scale.
    
    TM helices are typically:
    - 18-25 residues long
    - Highly hydrophobic (positive Kyte-Doolittle score)
    - Predominantly L, I, V, F, A residues
    """
    # Kyte-Doolittle hydrophobicity scale
    kd_scale = {
        'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
        'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
        'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
        'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
    }
    
    scores = []
    for i in range(len(sequence) - window_size + 1):
        window = sequence[i:i+window_size]
        score = sum(kd_scale.get(aa, 0) for aa in window) / window_size
        scores.append(score)
    
    # Find TM regions (hydrophobic stretches)
    tm_regions = []
    in_tm = False
    start = 0
    
    for i, score in enumerate(scores):
        if score > hydrophobicity_threshold and not in_tm:
            start = i
            in_tm = True
        elif score <= hydrophobicity_threshold and in_tm:
            if i - start >= 15:  # Minimum TM helix length
                tm_regions.append((start + 1, i + window_size))
            in_tm = False
    
    # Handle case where sequence ends in TM region
    if in_tm and len(scores) - start >= 15:
        tm_regions.append((start + 1, len(sequence)))
    
    return scores, tm_regions

hydro_scores, predicted_tm = predict_tm_regions(TARGET_SEQUENCE)

print(f"=== Hydrophobicity Analysis ===")
print(f"\nPredicted TM helices: {len(predicted_tm)}")
for i, (start, end) in enumerate(predicted_tm, 1):
    tm_seq = TARGET_SEQUENCE[start-1:end]
    print(f"  TM{i}: {start}-{end} ({end-start+1} aa): {tm_seq[:20]}...")

In [ ]:
# Visualize hydrophobicity plot
fig, ax = plt.subplots(figsize=(14, 5))

positions = range(1, len(hydro_scores)+1)
ax.plot(positions, hydro_scores, 'b-', linewidth=1)
ax.axhline(y=0.4, color='red', linestyle='--', linewidth=2, label='TM threshold (0.4)')
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)

# Shade TM regions
for start, end in predicted_tm:
    ax.axvspan(start, end, alpha=0.3, color='orange', label='Predicted TM' if start == predicted_tm[0][0] else '')

ax.fill_between(positions, hydro_scores, 0.4, 
                where=[s > 0.4 for s in hydro_scores], alpha=0.3, color='blue')

ax.set_xlabel('Residue Position', fontsize=12)
ax.set_ylabel('Hydrophobicity (Kyte-Doolittle)', fontsize=12)
ax.set_title(f'Hydrophobicity Plot - {len(predicted_tm)} TM helices predicted', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim(1, len(TARGET_SEQUENCE))

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / 'hydrophobicity_plot.png', dpi=150)
plt.show()

print(f"\nSaved: {OUTPUT_DIR}/hydrophobicity_plot.png")

In [ ]:
# Method 2: BLAST against PDB to identify fold
def blast_search(fasta_file, database, output_file, num_iterations=3):
    """Run PSI-BLAST search."""
    cmd = [
        "psiblast",
        "-query", str(fasta_file),
        "-db", database,
        "-out", str(output_file),
        "-outfmt", "6 sacc bitscore evalue pident qcovs qstart qend sstart send stitle",
        "-num_iterations", str(num_iterations),
        "-evalue", "0.001",
        "-max_target_seqs", "20"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return Path(output_file)

# Search PDB for structural homologs
pdb_blast = Path(OUTPUT_DIR) / "blast_pdb.out"
blast_search(TARGET_FASTA, PDBAA_DB, pdb_blast)

# Search SwissProt for functional annotation
sp_blast = Path(OUTPUT_DIR) / "blast_swissprot.out"
blast_search(TARGET_FASTA, SWISSPROT_DB, sp_blast)

print(f"BLAST results saved to: {pdb_blast}, {sp_blast}")

In [ ]:
# Display BLAST results
def load_blast_results(filepath):
    """Load and display BLAST results."""
    if filepath.exists() and filepath.stat().st_size > 0:
        df = pd.read_csv(filepath, sep='\t', header=None,
                        names=['Subject', 'BitScore', 'E-value', 'Identity', 'Coverage',
                               'QStart', 'QEnd', 'SStart', 'SEnd', 'Title'])
        return df
    return pd.DataFrame()

print("=== PDB BLAST Results (Top Templates) ===")
pdb_results = load_blast_results(pdb_blast)
display(pdb_results.head(10))

print("\n=== SwissProt BLAST Results (Functional Annotation) ===")
sp_results = load_blast_results(sp_blast)
display(sp_results.head(10))

In [ ]:
# Fold identification answer
fold_answer = f"""
=== ANSWER a) Fold Identification ===

HYDROPHOBICITY ANALYSIS:
- Number of predicted TM helices: {len(predicted_tm)}
- TM regions: {predicted_tm}

BLAST ANALYSIS:
- Top PDB hit: {pdb_results.iloc[0]['Subject'] if len(pdb_results) > 0 else 'N/A'}
- Identity: {pdb_results.iloc[0]['Identity'] if len(pdb_results) > 0 else 'N/A'}%
- Protein name: {pdb_results.iloc[0]['Title'].split('[')[0] if len(pdb_results) > 0 else 'N/A'}

FOLD TYPE:
Based on the analysis, this protein belongs to the:
[FILL IN: e.g., "Aquaporin/MIP fold - characterized by 6 TM helices plus 2 half-helices"]

STRUCTURAL FEATURES:
- Alpha-helical TM bundle (NOT beta-barrel)
- [FILL IN: specific features of this fold]

WEB RESOURCES FOR VERIFICATION:
- TMHMM: https://services.healthtech.dtu.dk/service.php?TMHMM-2.0
- Phobius: https://phobius.sbc.su.se/
- OPM (Orientation of Proteins in Membranes): https://opm.phar.umich.edu/
"""

print(fold_answer)

---
## b) Function and PFAM Family (0.5 pts)

### Biological Background: TM Protein Families

**MIP (Major Intrinsic Protein) family:**
- Pfam: PF00230
- Includes aquaporins and glycerol facilitators
- Function: Passive transport of water and small solutes
- Highly conserved NPA motifs in half-helices

**How to identify function:**
1. BLAST hits tell you what the protein is similar to
2. Pfam domain tells you the protein family
3. UniProt annotation provides detailed function

In [ ]:
# Search Pfam database
def search_pfam(fasta_file, output_file):
    """Search against local Pfam database."""
    cmd = [
        "hmmscan",
        "--tblout", str(output_file),
        "--domtblout", str(output_file).replace('.out', '_dom.out'),
        "-E", "1e-5",
        PFAM_DB,
        str(fasta_file)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return Path(output_file)

pfam_output = Path(OUTPUT_DIR) / "pfam_search.out"
search_pfam(TARGET_FASTA, pfam_output)

print(f"Pfam results: {pfam_output}")

In [ ]:
# Parse Pfam results
def parse_hmmscan(filepath):
    """Parse hmmscan tblout format."""
    results = []
    if filepath.exists():
        with open(filepath) as f:
            for line in f:
                if not line.startswith('#') and line.strip():
                    parts = line.split()
                    if len(parts) >= 10:
                        results.append({
                            'Family': parts[0],
                            'Accession': parts[1],
                            'E-value': float(parts[4]),
                            'Score': float(parts[5]),
                            'Description': ' '.join(parts[18:]) if len(parts) > 18 else ''
                        })
    return pd.DataFrame(results)

pfam_df = parse_hmmscan(pfam_output)
print("=== Pfam Domain Results ===")
display(pfam_df)

In [ ]:
# Download HMM profile
hmm_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}b.hmm"

def extract_hmm_profile(pfam_accession, output_file):
    """Extract HMM profile from local Pfam database."""
    # Remove version number
    acc_clean = pfam_accession.split('.')[0]
    
    cmd = ["hmmfetch", "-o", str(output_file), PFAM_DB, acc_clean]
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0 and output_file.exists():
        print(f"Saved HMM profile: {output_file}")
        return True
    else:
        print(f"Could not extract {pfam_accession}. Try downloading from:")
        print(f"  https://www.ebi.ac.uk/interpro/entry/pfam/{acc_clean}")
        return False

if len(pfam_df) > 0:
    best_pfam = pfam_df.iloc[0]['Accession']
    extract_hmm_profile(best_pfam, hmm_output)
    print(f"\nIdentified PFAM family: {pfam_df.iloc[0]['Family']} ({best_pfam})")
    print(f"Description: {pfam_df.iloc[0]['Description']}")
else:
    print("No Pfam domains found. Check sequence or try web search.")

In [ ]:
# Function and family answer
function_answer = f"""
=== ANSWER b) Function and PFAM Family ===

PFAM FAMILY:
- Family name: {pfam_df.iloc[0]['Family'] if len(pfam_df) > 0 else 'N/A'}
- Accession: {pfam_df.iloc[0]['Accession'] if len(pfam_df) > 0 else 'N/A'}
- E-value: {pfam_df.iloc[0]['E-value'] if len(pfam_df) > 0 else 'N/A'}

FUNCTION (from SwissProt BLAST):
- Top hit annotation: {sp_results.iloc[0]['Title'] if len(sp_results) > 0 else 'N/A'}

BIOLOGICAL FUNCTION:
[FILL IN based on results, e.g.:
"This protein is an aquaporin (water channel) belonging to the MIP family.
Aquaporins facilitate rapid, selective transport of water across membranes.
They are essential for osmotic regulation in bacteria and eukaryotes."]

HMM PROFILE SAVED: {hmm_output}
"""

print(function_answer)

---
## c) Structure Modeling (0.5 pts)

### Biological Background: TM Protein Modeling

**Challenges of TM protein modeling:**
- Fewer template structures available
- Need to maintain proper TM geometry
- Loop regions between TM helices can vary

**MODELLER for TM proteins:**
- Use templates from same family
- High sequence identity templates work best
- May need to model oligomeric state

In [ ]:
# Download best template
def download_pdb(pdb_id, output_dir=TEMPLATES_DIR):
    """Download PDB file from RCSB."""
    os.makedirs(output_dir, exist_ok=True)
    pdb_code = pdb_id.split('_')[0][:4].lower()
    output_path = Path(output_dir) / f"{pdb_code}.pdb"
    
    if not output_path.exists():
        import urllib.request
        url = f"https://files.rcsb.org/download/{pdb_code}.pdb"
        try:
            urllib.request.urlretrieve(url, output_path)
            print(f"Downloaded: {output_path}")
        except Exception as e:
            print(f"Download failed: {e}")
            return None
    else:
        print(f"Already exists: {output_path}")
    return output_path

# Get best template from BLAST results
if len(pdb_results) > 0:
    best_template_id = pdb_results.iloc[0]['Subject']
    best_template_pdb = download_pdb(best_template_id)
    template_identity = pdb_results.iloc[0]['Identity']
    print(f"\nBest template: {best_template_id}")
    print(f"Sequence identity: {template_identity}%")
else:
    print("No templates found. Try manual BLAST search at NCBI.")
    best_template_id = None

In [ ]:
# Create MODELLER alignment and script
def create_modeller_files_tm(target_seq, target_id, template_pdb, template_id, output_dir):
    """
    Create PIR alignment and MODELLER script for TM protein.
    """
    # Get template sequence
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('template', template_pdb)
    ppb = PPBuilder()
    
    template_seq = ""
    template_chain = 'A'
    for model in structure:
        for chain in model:
            for pp in ppb.build_peptides(chain):
                if len(str(pp.get_sequence())) > len(template_seq):
                    template_seq = str(pp.get_sequence())
                    template_chain = chain.id
    
    template_code = template_id.split('_')[0].lower()
    
    # Create PIR alignment
    pir_content = f""">P1;{template_code}
structureX:{template_code}:1:{template_chain}:+{len(template_seq)}:{template_chain}::::
{template_seq}*

>P1;{target_id}
sequence:{target_id}::::::::
{target_seq}*
"""
    
    pir_file = Path(output_dir) / f"{target_id}_alignment.pir"
    with open(pir_file, 'w') as f:
        f.write(pir_content)
    
    # Create MODELLER script
    script_content = f"""# MODELLER script for TM protein modeling
from modeller import *
from modeller.automodel import *

log.verbose()
env = Environ()

# Directories for input files
env.io.atom_files_directory = ['.', '{TEMPLATES_DIR}']

class MyModel(AutoModel):
    def special_patches(self, aln):
        # Rename MSE (selenomethionine) to MET if present in template
        self.rename_segments(segment_ids=['A'], renumber_residues=[1])

a = MyModel(env,
            alnfile='{pir_file}',
            knowns='{template_code}',
            sequence='{target_id}',
            assess_methods=(assess.DOPE, assess.GA341))

a.starting_model = 1
a.ending_model = 5  # Generate 5 models

# Optimization settings (good for TM proteins)
a.md_level = refine.slow

a.make()
"""
    
    script_file = Path(output_dir) / f"model_{target_id}.py"
    with open(script_file, 'w') as f:
        f.write(script_content)
    
    return pir_file, script_file, template_code

# Create MODELLER files
if best_template_pdb:
    pir_file, script_file, template_code = create_modeller_files_tm(
        TARGET_SEQUENCE, TARGET_NAME, best_template_pdb, best_template_id, MODELLER_DIR
    )
    print(f"Created alignment: {pir_file}")
    print(f"Created script: {script_file}")

In [ ]:
# MODELLER execution commands
model_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}c.pdb"

print("="*60)
print("MODELLER EXECUTION COMMANDS")
print("="*60)
print(f"""
Run these commands to build the TM protein model:

# Activate conda environment
conda activate AlphaBald

# Navigate to modeller directory
cd {MODELLER_DIR}

# Run MODELLER
python model_{TARGET_NAME}.py

# Alternative: Run with mod10.4 directly
mod10.4 model_{TARGET_NAME}.py

# Best model will be named:
#   {TARGET_NAME}.B99990001.pdb (lowest DOPE score)

# Copy best model to output
cp {MODELLER_DIR}/{TARGET_NAME}.B99990001.pdb {model_output}

NOTE: If sequence identity is very high (>95%), you might also use:
- Swiss-Model: https://swissmodel.expasy.org/
- Simply mutating the template structure
""")

---
## d) ProSA Validation Discussion (0.5 pts)

### Biological Background: Why ProSA Fails for TM Proteins

**ProSA statistical potentials:**
- Derived from ~3000 structures in PDB
- Predominantly SOLUBLE proteins
- Assumes surrounding WATER environment

**Problem for TM proteins:**
- TM regions are surrounded by LIPID, not water
- Hydrophobic residues (L, I, V, F) are STABILIZING in membrane
- ProSA sees these as "unfavorable" because in soluble proteins they'd be buried

**What ProSA will show (incorrectly):**
- High energy in TM regions
- Poor Z-score
- False positive "problematic" regions

In [ ]:
prosa_output = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}d.png"

prosa_discussion = f"""
========================================================================
ANSWER d) ProSA Validation - Critical Discussion for TM Proteins
========================================================================

QUESTION: Can ProSA be used to validate transmembrane proteins?

ANSWER: **NO - ProSA is NOT appropriate for TM proteins**

REASONS:

1. STATISTICAL POTENTIALS ARE ENVIRONMENT-SPECIFIC
   - ProSA potentials derived from SOLUBLE proteins in AQUEOUS environment
   - TM proteins exist in LIPID bilayer environment
   - Energy functions are fundamentally different

2. HYDROPHOBICITY INTERPRETATION IS INVERTED
   - Soluble proteins: Hydrophobic residues buried in core = FAVORABLE
   - TM proteins: Hydrophobic residues exposed to lipid = FAVORABLE
   - ProSA misinterprets exposed hydrophobic surfaces as unfavorable

3. EXPECTED ProSA RESULTS FOR THIS PROTEIN:
   - Z-score likely OUTSIDE normal range (too positive)
   - High energy peaks at TM helix positions: {predicted_tm}
   - These are FALSE POSITIVES - the TM regions are actually correct!

BETTER ALTERNATIVES FOR TM PROTEIN VALIDATION:

1. **QMEANBrane** (Swiss-Model)
   - Specifically designed for membrane proteins
   - URL: https://swissmodel.expasy.org/qmean/
   - Provides membrane-aware quality scores

2. **MolProbity** (Ramachandran, rotamers)
   - Backbone geometry is universal
   - URL: http://molprobity.biochem.duke.edu/
   - Good for local stereochemistry

3. **PROCHECK** (stereochemistry)
   - Bond lengths, angles, torsions
   - Not environment-dependent

4. **OPM database** (membrane orientation)
   - Validates TM helix positioning
   - URL: https://opm.phar.umich.edu/

5. **TOPCONS** (topology validation)
   - Compares predicted vs model topology
   - URL: https://topcons.cbr.su.se/

FOR THE EXAM:
Still submit ProSA results (as requested), BUT clearly note in your answer
that the high-energy regions at TM helices are EXPECTED FALSE POSITIVES
due to ProSA's soluble protein bias.
"""

print(prosa_discussion)

# Save discussion to file
with open(Path(OUTPUT_DIR) / "prosa_discussion.txt", 'w') as f:
    f.write(prosa_discussion)

In [ ]:
print(f"""
=== ProSA Submission Instructions ===

1. Go to ProSA web server:
   https://prosa.services.came.sbg.ac.at/prosa.php

2. Upload your model: {model_output}

3. Screenshot/download the results

4. Save energy profile as: {prosa_output}

5. In your answer, include:
   - The Z-score value
   - Statement that ProSA is NOT suitable for TM proteins
   - Explanation of why TM regions show high energy (false positives)
   - Alternative validation methods you would use instead
""")

---
## e) Oligomeric State Analysis (1.0 pts)

### Biological Background: TM Protein Oligomerization

**Why TM proteins oligomerize:**
- Functional channels/pores require multiple subunits
- Stability in membrane enhanced by oligomerization
- Cooperative binding/regulation

**Aquaporins form TETRAMERS:**
- Each protomer has its own water pore
- Central pore between 4 subunits may transport gases
- Tetramer is the biological assembly

**GxxxG motifs:**
- (G/A/S)xxxG sequence pattern
- Common TM helix-helix interaction motif
- Allows close helix packing

In [ ]:
oligomer_image = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}e.jpg"
oligomer_pdb = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}e.pdb"

# Search for GxxxG and related dimerization motifs
def find_gxxxg_motifs(sequence, tm_regions):
    """
    Find GxxxG and similar small-xxx-small motifs.
    These mediate TM helix-helix interactions.
    """
    patterns = [
        (r'G...G', 'GxxxG'),
        (r'G...A', 'GxxxA'),
        (r'A...G', 'AxxxG'),
        (r'A...A', 'AxxxA'),
        (r'S...G', 'SxxxG'),
        (r'G...S', 'GxxxS'),
    ]
    
    all_motifs = []
    for pattern, name in patterns:
        for match in re.finditer(pattern, sequence):
            pos = match.start() + 1
            in_tm = any(start <= pos <= end for start, end in tm_regions)
            all_motifs.append({
                'Motif': name,
                'Position': pos,
                'Sequence': match.group(),
                'In_TM': 'Yes' if in_tm else 'No'
            })
    
    return pd.DataFrame(all_motifs)

motifs_df = find_gxxxg_motifs(TARGET_SEQUENCE, predicted_tm)
print("=== TM Dimerization Motifs ===")
print("\nGxxxG-like motifs mediate helix-helix interactions in membranes.")
print("Motifs within TM regions are most relevant for oligomerization.\n")
display(motifs_df[motifs_df['In_TM'] == 'Yes'])

In [ ]:
# Check biological assembly from template
print("""=== How to Determine Oligomeric State ==="""

1. CHECK PDB BIOLOGICAL ASSEMBLY:
   - Go to RCSB PDB: https://www.rcsb.org/structure/{best_template_id.split('_')[0] if best_template_id else 'XXXX'}
   - Look at 'Biological Assembly' section
   - Aquaporins typically show TETRAMER

2. CHECK UniProt ANNOTATION:
   - Search UniProt for protein name
   - Look under 'Subunit' section
   - Usually states "Homotetramer" for aquaporins

3. DOWNLOAD BIOLOGICAL ASSEMBLY:
   - On PDB page, click 'Download Files'
   - Select 'PDB Format' > 'Biological Assembly'
   - This gives the functional oligomer

4. SWISS-MODEL OLIGOMER MODELING:
   - Swiss-Model can build oligomers automatically
   - URL: https://swissmodel.expasy.org/
   - Select template with biological assembly

""")

In [ ]:
oligomer_answer = f"""
=== ANSWER e) Oligomeric State ===

GxxxG MOTIF ANALYSIS:
- Total GxxxG-like motifs found: {len(motifs_df)}
- Motifs in TM regions: {len(motifs_df[motifs_df['In_TM'] == 'Yes'])}
- These motifs suggest potential for oligomerization

TEMPLATE ANALYSIS:
- Best template: {best_template_id if best_template_id else 'N/A'}
- [CHECK PDB biological assembly for this structure]

OLIGOMERIC STATE:
[FILL IN based on template analysis, e.g.:
"Aquaporins function as HOMOTETRAMERS.
- Each monomer forms an independent water pore
- The tetramer is stabilized by helix-helix interactions at the interface
- GxxxG motifs found in TM regions mediate these interactions"]

MODELING APPROACH:
1. Downloaded biological assembly from PDB (tetramer)
2. Used Swiss-Model with oligomeric template
3. Saved tetramer model as: {oligomer_pdb}

VISUALIZATION:
- Created figure showing tetramer from membrane plane
- Saved as: {oligomer_image}
"""

print(oligomer_answer)

In [ ]:
# Commands for oligomer modeling
print("="*60)
print("OLIGOMER MODELING COMMANDS")
print("="*60)
print(f"""
Option 1: Download biological assembly

# Get tetramer from PDB (example for aquaporin 1RC2)
wget https://files.rcsb.org/download/{best_template_id.split('_')[0].lower() if best_template_id else 'XXXX'}-assembly1.cif

Option 2: Swiss-Model web server

1. Go to https://swissmodel.expasy.org/
2. Paste sequence or upload FASTA
3. Click 'Build Model'
4. Select template with quaternary structure
5. Download oligomeric model

Option 3: MODELLER with symmetry (advanced)

# Create symmetric tetramer using MODELLER
# Requires template biological assembly

PyMOL commands for visualization:

load {oligomer_pdb}, tetramer
bg_color white
set_view (...)  # Set membrane view
color red, chain A
color blue, chain B  
color green, chain C
color yellow, chain D
show cartoon
ray 1200, 900
png {oligomer_image}
""")

---
## f) Fix Structural Problems (1.0 pts)

### Strategy for TM Protein Refinement

**Common problems in TM models:**
1. Loop conformations (most variable)
2. Side chain rotamers
3. Helix tilts

**Refinement approach:**
1. Identify problems using MolProbity
2. Refine loops with MODELLER
3. Energy minimize in membrane environment (CHARMM-GUI)

In [ ]:
fixed_monomer = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}f1.pdb"
fixed_complex = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}f2.pdb"
fixed_image = Path(OUTPUT_DIR) / f"{OUTPUT_PREFIX}f.jpg"

print(f"""
=== Structure Refinement Protocol ===

STEP 1: IDENTIFY PROBLEMS

A. MolProbity Analysis (http://molprobity.biochem.duke.edu/)
   - Upload model: {model_output}
   - Check Ramachandran outliers
   - Check rotamer outliers  
   - Check clashscore

B. WHAT-CHECK or similar
   - Bond length/angle deviations
   - Unusual contacts

STEP 2: LOOP REFINEMENT (if needed)

# Identify loop regions (between TM helices)
Loop regions based on TM prediction:
""")

# Calculate loop regions
loops = []
if len(predicted_tm) > 1:
    for i in range(len(predicted_tm) - 1):
        loop_start = predicted_tm[i][1] + 1
        loop_end = predicted_tm[i+1][0] - 1
        if loop_end > loop_start:
            loops.append((loop_start, loop_end))
            print(f"  Loop {i+1}: {loop_start}-{loop_end} ({loop_end-loop_start+1} residues)")

In [ ]:
# Create loop refinement script
def create_tm_loop_refinement_script(model_file, target_id, loop_regions, output_dir):
    """Create MODELLER script for TM protein loop refinement."""
    
    loop_selection = ""
    for start, end in loop_regions:
        loop_selection += f"            self.residue_range('{start}:', '{end}:'),\n"
    
    script = f"""# MODELLER Loop Refinement for TM Protein
from modeller import *
from modeller.automodel import *

log.verbose()
env = Environ()

env.io.atom_files_directory = ['.', '{TEMPLATES_DIR}', '{MODELLER_DIR}', '{OUTPUT_DIR}']

class TMLoopModel(LoopModel):
    def select_loop_atoms(self):
        # Select loop regions between TM helices
        return Selection(
{loop_selection}        )

m = TMLoopModel(env,
                inimodel='{model_file}',
                sequence='{target_id}')

m.loop.starting_model = 1
m.loop.ending_model = 20  # Generate more models for loops
m.loop.md_level = refine.slow  # Thorough refinement

m.make()
"""
    
    script_file = Path(output_dir) / f"refine_loops_{target_id}.py"
    with open(script_file, 'w') as f:
        f.write(script)
    
    return script_file

if loops:
    refine_script = create_tm_loop_refinement_script(
        str(model_output), TARGET_NAME, loops, MODELLER_DIR
    )
    print(f"Created loop refinement script: {refine_script}")

In [ ]:
print("="*60)
print("REFINEMENT EXECUTION COMMANDS")
print("="*60)
print(f"""
STEP 2: RUN LOOP REFINEMENT

conda activate AlphaBald
cd {MODELLER_DIR}
python refine_loops_{TARGET_NAME}.py

# Best refined model: {TARGET_NAME}.BL00200001.pdb

STEP 3: MEMBRANE ENVIRONMENT MINIMIZATION (Optional but recommended)

A. CHARMM-GUI (https://www.charmm-gui.org/)
   1. Go to Input Generator > Membrane Builder
   2. Upload your model
   3. Let it auto-orient using OPM
   4. Build membrane system
   5. Download GROMACS/NAMD files

B. Local minimization with GROMACS:
   gmx grompp -f minim.mdp -c system.gro -p topol.top -o em.tpr
   gmx mdrun -v -deffnm em

STEP 4: VALIDATE REFINED MODEL

# Re-run QMEANBrane to check improvement
# Compare Ramachandran before/after

STEP 5: SAVE OUTPUTS

cp refined_monomer.pdb {fixed_monomer}
cp refined_tetramer.pdb {fixed_complex}

# Create comparison figure in PyMOL
load {model_output}, original
load {fixed_monomer}, refined
align refined, original
color red, original
color green, refined
ray 1200, 900
png {fixed_image}
""")

---
## Output Files Checklist

In [ ]:
expected_outputs = [
    (f"{OUTPUT_PREFIX}b.hmm", "PFAM HMM profile (MIP family)"),
    (f"{OUTPUT_PREFIX}c.pdb", "Initial monomer structure model"),
    (f"{OUTPUT_PREFIX}d.png", "ProSA energy profile (with discussion!)"),
    (f"{OUTPUT_PREFIX}e.jpg", "Oligomeric state visualization"),
    (f"{OUTPUT_PREFIX}e.pdb", "Oligomer/tetramer structure"),
    (f"{OUTPUT_PREFIX}f1.pdb", "Corrected monomer"),
    (f"{OUTPUT_PREFIX}f2.pdb", "Corrected oligomer/complex"),
    (f"{OUTPUT_PREFIX}f.jpg", "Before/after comparison image"),
    ("prosa_discussion.txt", "ProSA suitability discussion"),
    ("hydrophobicity_plot.png", "TM helix prediction"),
]

print(f"=== Output Files Checklist ({OUTPUT_DIR}/) ===")
print("\nCheck each file as you complete it:\n")
for filename, description in expected_outputs:
    filepath = Path(OUTPUT_DIR) / filename
    status = "[x]" if filepath.exists() else "[ ]"
    print(f"{status} {filename:25s} - {description}")